# Linear Discriminant Analysis (LDA) and Quadratic Discriminant Analysis (QDA)

## 1. Definition
LDA and QDA are probabilistic classifiers derived from Bayes' rule and class-conditional Gaussian distributions.

- **LDA:** each class has its own mean but all classes share one covariance matrix. This produces linear decision boundaries.
- **QDA:** each class has its own mean and covariance matrix. This produces quadratic decision boundaries.

## 2. Derivation
Bayes' rule gives $P(y=k\mid x)\propto P(y=k)p(x\mid y=k)$. Suppose $x\mid y=k\sim\mathcal N(\mu_k,\Sigma_k)$ and class prior is $\pi_k$. The discriminant score is
$$
\delta_k(x)=\log\pi_k-\frac12\log|\Sigma_k|-\frac12(x-\mu_k)^T\Sigma_k^{-1}(x-\mu_k).
$$
If $\Sigma_k=\Sigma$ for every class, the shared quadratic term $x^T\Sigma^{-1}x$ cancels when comparing classes, leaving a linear function of x (LDA). If covariances differ, that quadratic term no longer cancels (QDA).

## 3. Why it works and assumptions
If class-conditional Gaussian assumptions and prior estimates are reasonable, these scores are Bayes-derived decision rules. LDA can be statistically more stable because it estimates one pooled covariance; QDA is more flexible but estimates more parameters. The Gaussian assumption can fail; covariance estimates can be singular when features are numerous or collinear.

## 4. Dataset

Use a multiclass classification dataset with standardized features. Standardization is not mathematically required for Gaussian discriminant models, but it can help numerical conditioning. Fit scaling on the training split only.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
X, y = make_classification(n_samples=1200, n_features=8, n_informative=6, n_redundant=1,
                           n_classes=3, n_clusters_per_class=1, class_sep=1.15, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
classes = np.unique(y_train)
print("Train class counts:", np.bincount(y_train))


Train class counts: [301 301 298]


## 5. From-scratch Gaussian discriminant classifier

We estimate class means, class priors and covariance matrices. A small diagonal regularizer makes covariance solves more stable. The LDA estimator uses a pooled covariance; QDA estimates one covariance per class.

In [2]:
class GaussianDiscriminantScratch:
    def __init__(self, mode="lda", reg=1e-5):
        if mode not in {"lda", "qda"}:
            raise ValueError("mode must be 'lda' or 'qda'")
        if reg < 0:
            raise ValueError("reg must be nonnegative")
        self.mode, self.reg = mode, reg

    def fit(self, X, y):
        X = np.asarray(X, dtype=float); y = np.asarray(y)
        if X.ndim != 2 or y.ndim != 1 or len(X) != len(y) or len(X) == 0:
            raise ValueError("X must be non-empty 2D and y matching 1D")
        self.classes_, counts = np.unique(y, return_counts=True)
        self.priors_ = counts / counts.sum()
        self.means_ = np.stack([X[y == c].mean(axis=0) for c in self.classes_])
        d = X.shape[1]
        if self.mode == "lda":
            centered = np.concatenate([X[y == c] - self.means_[i] for i, c in enumerate(self.classes_)])
            cov = centered.T @ centered / max(len(X)-len(self.classes_), 1)
            self.covs_ = np.repeat((cov + self.reg*np.eye(d))[None], len(self.classes_), axis=0)
        else:
            covs = []
            for i, c in enumerate(self.classes_):
                Xi = X[y == c] - self.means_[i]
                covs.append(Xi.T @ Xi / max(len(Xi)-1, 1) + self.reg*np.eye(d))
            self.covs_ = np.asarray(covs)
        return self

    def predict_log_proba(self, X):
        X = np.asarray(X, dtype=float)
        n, d = X.shape
        scores = []
        for i, c in enumerate(self.classes_):
            cov = self.covs_[i]
            sign, logdet = np.linalg.slogdet(cov)
            if sign <= 0:
                raise ValueError("Covariance is not positive definite; increase reg")
            diff = X - self.means_[i]
            sol = np.linalg.solve(cov, diff.T).T
            quad = np.einsum("ij,ij->i", diff, sol)
            scores.append(np.log(self.priors_[i]) - 0.5*logdet - 0.5*quad)
        scores = np.column_stack(scores)
        m = scores.max(axis=1, keepdims=True)
        lse = m + np.log(np.exp(scores-m).sum(axis=1, keepdims=True))
        return scores - lse

    def predict_proba(self, X):
        return np.exp(self.predict_log_proba(X))

    def predict(self, X):
        return self.classes_[self.predict_log_proba(X).argmax(axis=1)]

for mode in ("lda", "qda"):
    model = GaussianDiscriminantScratch(mode=mode, reg=1e-4).fit(X_train, y_train)
    pred = model.predict(X_test)
    print(mode.upper(), "scratch accuracy:", round(accuracy_score(y_test, pred), 4))
    assert np.allclose(model.predict_proba(X_test).sum(axis=1), 1, atol=1e-7)


LDA scratch accuracy: 0.9333
QDA scratch accuracy: 0.98


## 6. Compare with scikit-learn

Default solvers, covariance estimators, and regularization conventions may differ. Compare accuracy and probability validity; tune shrinkage or regularization for your data.

In [3]:
models = [
    ("LDA scratch", GaussianDiscriminantScratch("lda", reg=1e-4).fit(X_train, y_train), None),
    ("QDA scratch", GaussianDiscriminantScratch("qda", reg=1e-4).fit(X_train, y_train), None),
    ("LDA sklearn", LinearDiscriminantAnalysis(), "lda"),
    ("QDA sklearn", QuadraticDiscriminantAnalysis(reg_param=0.001), "qda")
]
for name, model, kind in models:
    if kind is None:
        pred = model.predict(X_test)
    else:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
    print(name, "accuracy:", round(accuracy_score(y_test, pred), 4))


LDA scratch accuracy: 0.9333
QDA scratch accuracy: 0.98
LDA sklearn accuracy: 0.9333
QDA sklearn accuracy: 0.9833


## 7. Hyperparameters, complexity, scaling

- **LDA:** pooled covariance estimation and optional shrinkage improve stability in high dimensions.
- **QDA:** `reg_param` shrinks class covariance estimates; greater flexibility needs more data.
- Fitting sample means costs $O(nd)$. Covariance estimation is roughly $O(nd^2)$; covariance factorization is $O(Kd^3)$. Inference with full covariance costs $O(md^2)$ after factorization, for m samples.
- QDA stores $O(Kd^2)$ covariance values; LDA stores $O(d^2)$ shared covariance.
- When d is large relative to class sample counts, covariance estimation can be unreliable. Use shrinkage, feature selection, PCA fitted within the training pipeline, or other classifiers.
- GPU: covariance calculations parallelize, but for small tabular data CPU overhead is often lower. Use batched linear algebra only if data and implementation warrant it.

## 8. Best practices and synthesis
LDA buys stability through a shared covariance assumption; QDA trades that assumption for a more flexible quadratic boundary. The key question is whether each class truly needs a separate covariance matrix and whether enough samples exist to estimate it well.

In [4]:
for mode in ("lda", "qda"):
    model = GaussianDiscriminantScratch(mode=mode, reg=1e-4).fit(X_train, y_train)
    assert model.predict(X_test).shape == y_test.shape
    assert np.isfinite(model.predict_proba(X_test)).all()
try:
    GaussianDiscriminantScratch(mode="invalid").fit(X_train, y_train)
except ValueError:
    pass
else:
    raise AssertionError("Expected invalid mode to raise")
print("All LDA/QDA checks passed.")


All LDA/QDA checks passed.
